In [ ]:
# vapor-eyes — shared config (%run ./config_nb). Permian/Delaware-Basin methane
# cascade on the GeoBrix lightweight tier: sets catalog/schema + the Volume ETL
# tree + toggles + downloaders. Mirrors the eo-series / helios config_nb (comments
# + minimal prints, no markdown cells).

# -- GeoBrix lightweight tier (option-1, default). Two-step install so a freshly
#    rebuilt same-version wheel's bytes replace the cached install (step 1
#    --force-reinstall --no-deps) and the extras still resolve (step 2). Then a
#    %restart_python (next cell) loads the fresh bytes.
%pip install --quiet --disable-pip-version-check --force-reinstall --no-deps "geobrix @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.4.0-py3-none-any.whl"
%pip install --quiet "geobrix[light,stac,vizx] @ file:///Volumes/geospatial_docs/geobrix/sample-data/geobrix-0.4.0-py3-none-any.whl"
%pip install --quiet rich

In [ ]:
%restart_python

In [ ]:
import os

from pyspark.databricks.sql import functions as DBF
from pyspark.sql import functions as F
from pyspark.sql.types import *

# GeoBrix light tier (option-1 default). option-2 (heavyweight) commented.
from databricks.labs.gbx.pyrx import functions as rx  # rx.rst_* (raster) + pmtiles_agg
from databricks.labs.gbx.pyvx import functions as vx  # vx.st_*  (vector: MVT / pyramid)

# from databricks.labs.gbx.rasterx import functions as rx  # option-2: heavyweight
# from databricks.labs.gbx.vectorx import functions as vx  # option-2: heavyweight
rx.register(spark)  # gbx_rst_*, gbx_pmtiles_agg, ...
vx.register(spark)  # gbx_st_asmvt, gbx_st_asmvt_pyramid, ...

from databricks.labs.gbx.ds.register import register

register(spark)  # gtiff_gbx / geojson_gbx / netcdf_gbx / ...

In [ ]:
# ============================================================
# USER SETTINGS — edit these (everything below is wired off them)
# ============================================================

# Unity Catalog: a Volume named 'data' must exist under catalog/schema.
catalog_name = "geospatial_docs"
schema_name = "vapor_eyes"

# Toggles (overridable per-notebook right after %run ./config_nb):
FULL_AOI = False           # False -> SMALL demo AOI; True -> full Delaware Basin
FORCE_REBUILD = False       # True -> re-download / re-create tables (skip-guards off)
INTERACTIVE_PLOTS = False   # False -> static maps (GitHub-friendly); True -> MapLibre

# Delaware Basin AOI (coverage-verified). SMALL is the east-basin (TX) cluster
# around an EMIT super-emitter co-located with dense TX RRC wells; FULL is the
# wider basin. (minx, miny, maxx, maxy; EPSG:4326)
SMALL_BBOX = (-103.25, 31.30, -102.85, 31.62)
FULL_BBOX = (-103.60, 31.05, -102.60, 31.85)
def get_aoi_bbox():
    """Resolve the AOI bbox from the CURRENT FULL_AOI toggle — re-evaluated on every
    call (NOT cached at %run time), so flipping `FULL_AOI = True` in a cell right after
    `%run ./config_nb` takes effect everywhere the AOI is used. Same dynamic-override
    pattern as FORCE_REBUILD / INTERACTIVE_PLOTS. Use `get_aoi_bbox()` in the notebooks,
    never a cached AOI_BBOX variable."""
    return FULL_BBOX if FULL_AOI else SMALL_BBOX

# Datetime window (anchored on the 2023-07-31 EMIT overpass over the SMALL cluster).
DATE_WINDOW = "2023-07-15/2023-08-20"

# EMIT / NASA Earthdata token — a Unity Catalog secret (catalog.schema.name),
# read via the secret() SQL function. NB03 (EMIT) only; NB01/02/04/05 don't need it.
EARTHDATA_UC_SECRET = "geospatial_docs.vapor_eyes.earthdata_token"

In [ ]:
# Earthdata token -> env (guarded; NB03 prints clear guidance if absent). It's a
# Unity Catalog secret (catalog.schema.key), read with the 3-arg
# dbutils.secrets.get(catalog, schema, key) (value auto-redacted). The token is
# never printed either way.
_cat, _sch, _name = EARTHDATA_UC_SECRET.split(".")
_tok = None
try:
    _tok = dbutils.secrets.get(_cat, _sch, _name)  # UC secret (3-arg)
except Exception:
    _tok = None
if _tok:
    os.environ["EARTHDATA_TOKEN"] = _tok
    print(f"... EARTHDATA_TOKEN loaded from {EARTHDATA_UC_SECRET}")
else:
    print(
        f"... EARTHDATA_TOKEN NOT set. NB03 (EMIT) needs UC secret "
        f"{EARTHDATA_UC_SECRET}; NB01/02/04/05 run fine without it."
    )
_tok = None  # clear the value from the notebook namespace (env var retains it)

In [ ]:
# Spark-conf tuning, guarded for Serverless (no-ops there; AQE handles it).
def set_conf_safe(key, value):
    try:
        spark.conf.set(key, value)
        return True
    except Exception as e:
        print(f"... skipping spark.conf.set({key}) [Serverless?]: {type(e).__name__}")
        return False


set_conf_safe("spark.sql.adaptive.coalescePartitions.enabled", "false")
set_conf_safe("spark.sql.shuffle.partitions", 512)

In [ ]:
# Apply catalog/schema from USER SETTINGS.
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {schema_name}")
spark.sql(f"USE DATABASE {schema_name}")
print(f"... catalog: '{catalog_name}' (USE)")
print(f"... schema:  '{schema_name}' (CREATE / USE)")

In [ ]:
# Volume ETL tree (Volume 'data' must exist).
ETL_DIR = f"/Volumes/{catalog_name}/{schema_name}/data"
VAPOR_EYES_DIR = f"{ETL_DIR}/vapor-eyes"
S5P_DIR = f"{VAPOR_EYES_DIR}/s5p"
S2_DIR = f"{VAPOR_EYES_DIR}/sentinel2"
EMIT_DIR = f"{VAPOR_EYES_DIR}/emit"
WELLS_DIR = f"{VAPOR_EYES_DIR}/wells"
TILES_DIR = f"{VAPOR_EYES_DIR}/tiles"
for _d in (S5P_DIR, S2_DIR, EMIT_DIR, WELLS_DIR, TILES_DIR):
    dbutils.fs.mkdirs(_d)
print(f"... VAPOR_EYES_DIR: '{VAPOR_EYES_DIR}' (MKDIRS s5p/ sentinel2/ emit/ wells/ tiles/)")
print(f"... AOI ({'FULL' if FULL_AOI else 'SMALL'}): {get_aoi_bbox()}  (via get_aoi_bbox(); re-reads FULL_AOI each call)")
print(f"... DATE_WINDOW: {DATE_WINDOW}")
print(
    f"... toggles: FULL_AOI={FULL_AOI}  FORCE_REBUILD={FORCE_REBUILD}  "
    f"INTERACTIVE_PLOTS={INTERACTIVE_PLOTS}"
)

In [ ]:
# Idempotent managed-Delta materializer (Serverless-safe cache() stand-in).
def finalize_delta(df, tbl_name, do_display=True):
    if FORCE_REBUILD:
        spark.sql(f"DROP TABLE IF EXISTS {tbl_name}")
    elif spark.catalog.tableExists(tbl_name):
        if [f.name for f in spark.table(tbl_name).schema] != [
            f.name for f in df.schema
        ]:
            print(f"... schema changed for {tbl_name} -> rewriting (was stale)")
            spark.sql(f"DROP TABLE IF EXISTS {tbl_name}")
    if not spark.catalog.tableExists(tbl_name):
        df.write.mode("overwrite").saveAsTable(tbl_name)
        print(f"... wrote table {tbl_name} ({spark.table(tbl_name).count():,} rows)")
    else:
        print(f"... table {tbl_name} exists (skip; FORCE_REBUILD=False)")
    out = spark.table(tbl_name)
    if do_display:
        out.printSchema()
    return out

In [ ]:
# Downloaders + STAC client + vizx helpers.
from databricks.labs.gbx.sample import (
    EmitDownloader,
    TropomiDownloader,
    WellsDownloader,
)
from databricks.labs.gbx.stac import StacClient

tropomi = TropomiDownloader()
emit = EmitDownloader()
wells = WellsDownloader()
stac_client = StacClient()

from databricks.labs.gbx.vizx import (
    cells_as_gdf,
    grid_layer,
    plot_interactive,
    plot_pmtiles,
    plot_raster,
    plot_tile,
)

print(
    "... downloaders: tropomi, emit, wells | stac_client | "
    "vizx: plot_raster, plot_tile, cells_as_gdf, plot_pmtiles"
)

In [ ]:
# Static-vs-interactive view helpers, gated by INTERACTIVE_PLOTS (mirrors the helios
# series so the toggle behaves consistently across vapor-eyes). The numbered
# notebooks render through these:
#   show_cells   — H3 cell choropleth: static (cells_as_gdf + contextily basemap,
#                  zoomed out for regional context) or interactive MapLibre (grid_layer)
#   show_tile    — a raster tile draped over a basemap (plot_tile); static in BOTH
#                  modes (a raw in-memory tile has no tiled interactive form), the
#                  toggle honored for API symmetry (as helios does for COGs)
#   show_pmtiles — a PMTiles archive: static image (default) or interactive MapLibre


def show_cells(
    df, *, cellid_col, column, cmap="inferno", label=None, title=None,
    center=None, zoom=9,
):
    if INTERACTIVE_PLOTS:
        return plot_interactive(
            [
                grid_layer(
                    df, grid_system="h3", cellid_col=cellid_col, column=column,
                    cmap=cmap, opacity=0.7, label=label or column,
                )
            ],
            center=center, zoom=zoom,
        )
    import contextily as cx

    gdf = cells_as_gdf(
        df, cell_col=cellid_col, extra_cols=[column], max_rows=None
    ).to_crs(3857)
    ax = gdf.plot(
        column=column, legend=True, figsize=(9, 9), cmap=cmap,
        alpha=0.6, edgecolor="black", linewidth=0.2,
    )
    minx, miny, maxx, maxy = gdf.total_bounds
    pad = 0.8 * max(maxx - minx, maxy - miny)  # zoom out for regional context
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)
    ax.set_axis_off()
    if title:
        ax.set_title(title)
    return ax


def show_tile(tile, **kw):
    return plot_tile(tile, **kw)


def show_pmtiles(path, **kw):
    from databricks.labs.gbx.pmtiles import pmtiles_info

    info = pmtiles_info(path)
    print(
        f"... pmtiles: type={info.get('tile_type')} "
        f"zoom={info.get('min_zoom')}-{info.get('max_zoom')} bounds={info.get('bounds')}"
    )
    if INTERACTIVE_PLOTS:
        return plot_pmtiles(path, **kw)
    return plot_pmtiles(path, max_embed_mb=0, **kw)


print("... view helpers: show_cells, show_tile, show_pmtiles (gated by INTERACTIVE_PLOTS)")